# OmniMed-FL — reviewer experiments (no Google Drive)

Runs the experiments the reviewers asked for, entirely in the Colab session.

**Persistence matters:** `/content` is wiped when the session ends. Download `results_v2.json` after every chunk. After E4 starts, also keep `warmstart_results_v2_concat.pt`; it is used only by the explicitly labeled pooled-data oracle ablation.

That is why the run is split into resumable chunks instead of one long job. Runtime is hardware- and contention-dependent and may exceed a free-tier session, so download after each completed record group.

| Chunk | Experiments | Runtime | Answers |
|---|---|---|---|
| A | E1 α sweep, E2 client sweep | hardware-dependent | R3.1, R3.4 |
| B | E3 anti-collapse, E4 pooled-data oracle ablation | hardware-dependent | R3.2 |
| C | E8 matched baselines | hardware-dependent | R3.3 |
| D | E5 fusion seeds, E6 cost, E7 retrieval | hardware-dependent | R3.1, R1.2, R3.2 |

Chunks A and C use the operational FL initialization: public pretrained encoders plus random task heads, with no pooled-data training. Only E4 constructs a pooled-data oracle, explicitly as a non-deployable diagnostic.

## Order of operations

1. GPU on (cell 1) → deps (cell 2) → upload the 3 `.py` files (cell 3)
2. **Smoke test (cell 4)** — confirms every path before the standard run
3. Chunk A → **DOWNLOAD** → Chunk B → **DOWNLOAD** → Chunk C → **DOWNLOAD** → Chunk D → **DOWNLOAD**
4. Tables (last cell) → download `paper_assets/`

**If you get disconnected:** re-run cells 1–3, then the RESUME cell to upload your saved `results_v2.json`, then continue from the chunk you were on. Finished work is skipped automatically.

In [ ]:
#@title 1 — GPU check
import torch, subprocess
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi","--query-gpu=memory.total,memory.free","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 2 — extra dependency (faiss, needed by E7 only)
!pip -q install faiss-cpu 2>&1 | tail -1
print("ok")

In [ ]:
#@title 3 — upload the three .py files
import os, sys, shutil
os.chdir("/content")

NEED = ["MedFederate_Colab_Complete.py", "omnimed_experiments.py", "omnimed_make_tables.py"]
missing = [f for f in NEED if not os.path.exists(f"/content/{f}")]

if missing:
    print("Select these from your computer:", missing)
    print("(they are in Globecom_final/experiments/)")
    from google.colab import files
    for name in files.upload():
        shutil.move(name, f"/content/{name}")
    missing = [f for f in NEED if not os.path.exists(f"/content/{f}")]

assert not missing, f"still missing: {missing}"
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
print("all three present in /content")

In [ ]:
#@title 4 — SMOKE TEST. Do not skip; it catches breakage cheaply.
import importlib, omnimed_experiments as ox
importlib.reload(ox)

ox.main(base_py="/content/MedFederate_Colab_Complete.py",
        tier="smoke",
        out="/content/results_smoke.json")

print("\n>>> If this printed 'Done', the pipeline works. Move on to Chunk A.")

---
## Chunk A — α sweep and client sweep

In [ ]:
#@title Chunk A — E1 + E2   (hardware-dependent)
import importlib, omnimed_experiments as ox
importlib.reload(ox)

ox.main(base_py="/content/MedFederate_Colab_Complete.py",
        tier="standard",
        out="/content/results_v2.json",
        only=["E1", "E2"])

print("\n>>> CHUNK DONE — run the DOWNLOAD cell below before doing anything else.")

In [ ]:
#@title DOWNLOAD RESULTS — run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["results_v2.json", "warmstart_results_v2_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)

---
## Chunk B — anti-collapse and pooled-data oracle ablations

In [ ]:
#@title Chunk B — E3 + E4   (hardware-dependent)
import importlib, omnimed_experiments as ox
importlib.reload(ox)

ox.main(base_py="/content/MedFederate_Colab_Complete.py",
        tier="standard",
        out="/content/results_v2.json",
        only=["E3", "E4"])

print("\n>>> CHUNK DONE — run the DOWNLOAD cell below before doing anything else.")

In [ ]:
#@title DOWNLOAD RESULTS — run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["results_v2.json", "warmstart_results_v2_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)

---
## Chunk C — matched-setting federated baselines

In [ ]:
#@title Chunk C — E8   (hardware-dependent)
import importlib, omnimed_experiments as ox
importlib.reload(ox)

ox.main(base_py="/content/MedFederate_Colab_Complete.py",
        tier="standard",
        out="/content/results_v2.json",
        only=["E8"],
        alphas=[0.1])

print("\n>>> CHUNK DONE — run the DOWNLOAD cell below before doing anything else.")

In [ ]:
#@title DOWNLOAD RESULTS — run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["results_v2.json", "warmstart_results_v2_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)

---
## Chunk D — fusion variance, measured cost, retrieval

In [ ]:
#@title Chunk D — E5 + E6 + E7   (hardware-dependent)
import importlib, omnimed_experiments as ox
importlib.reload(ox)

ox.main(base_py="/content/MedFederate_Colab_Complete.py",
        tier="standard",
        out="/content/results_v2.json",
        only=["E5", "E6", "E7"])

print("\n>>> CHUNK DONE — run the DOWNLOAD cell below before doing anything else.")

In [ ]:
#@title DOWNLOAD RESULTS — run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["results_v2.json", "warmstart_results_v2_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)

---
## Resuming after a disconnect

In [ ]:
#@title RESUME - run this if you are continuing a previous session
# Upload results_v2.json. If E4 has started or completed, also upload its
# matching warmstart_results_v2_concat.pt. Operational E1/E2/E3/E8 runs never
# consume that checkpoint. If the JSON contains E4 oracle metrics but the
# checkpoint is missing, the suite stops rather than mixing initializations.
# Skip this cell on a fresh start.
from google.colab import files
import shutil, os
print("Select results_v2.json and, if present, warmstart_results_v2_concat.pt:")
for name in files.upload():
    shutil.move(name, f"/content/{name}")
    print("  restored", name)
have = [f for f in os.listdir("/content") if f.endswith((".json", ".pt"))]
print("\nfiles now in /content:", have)
if not any(f.endswith(".pt") for f in have):
    print("No E4 oracle checkpoint restored; this is fine before E4 begins.")


---
## Build the tables (after all four chunks)

In [ ]:
#@title Build LaTeX tables, figures and the readout
import importlib, sys, omnimed_make_tables as mt
importlib.reload(mt)

sys.argv = ["omnimed_make_tables.py",
            "--results", "/content/results_v2.json",
            "--outdir",  "/content/paper_assets"]
mt.main()

print("\n" + "="*70)
print(open("/content/paper_assets/SUMMARY.md").read())

In [ ]:
#@title Download paper_assets as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/paper_assets", "zip", "/content/paper_assets")
files.download("/content/paper_assets.zip")

## What to send back

`paper_assets.zip` and `results_v2.json`.

**Read `SUMMARY.md` first.** It flags where a result argues against a claim currently in the paper — two are live possibilities:

- If the `neither` arm of **E3** doesn't collapse, the anti-collapse stack isn't doing what the paper says it does. The claim changes, not the experiment.
- If **E8** shows FedProx or SCAFFOLD beating FedAvg at α=0.1, switch the aggregator and say so.